# Recurrent Neural Network (RNN)
## Time Series Prediction using Google Stock Prices Dataset

We will use the `yfinance` library to download historical Google stock prices and build an RNN to predict future closing prices.

In [ ]:
# Install yfinance if you don't have it installed by uncommenting the line below:
# !pip install yfinance

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, SimpleRNN, Dropout, Input

In [ ]:
# 1. Download Google Stock Data
# We will download data from Jan 2015 to Jan 2023
df = yf.download('GOOG', start='2015-01-01', end='2023-01-01')

print(df.head())

# Visualize the closing price history
plt.figure(figsize=(16, 6))
plt.title('Google Stock Close Price History')
plt.plot(df['Close'])
plt.xlabel('Date')
plt.ylabel('Close Price USD ($)')
plt.show()

In [ ]:
import numpy as np

# 2. Data Preprocessing
# Extract only the 'Close' column
data = df[['Close']].values

# Scale the data to be between 0 and 1
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(data)

# Determine the length of the training data (80%)
training_data_len = int(np.ceil(len(data) * 0.8))
print(f"Training Data Length: {training_data_len}")

In [ ]:
# Create the training dataset
train_data = scaled_data[0:int(training_data_len), :]

# Split the data into x_train and y_train
# We will use the past 60 days to predict the 61st day
x_train, y_train = [], []

for i in range(60, len(train_data)):
    x_train.append(train_data[i-60:i, 0])
    y_train.append(train_data[i, 0])

# Convert to numpy arrays
x_train, y_train = np.array(x_train), np.array(y_train)

# Reshape data to fit RNN (samples, time steps, features)
x_train = np.reshape(x_train, (x_train.shape[0], x_train.shape[1], 1))

print(f"x_train shape: {x_train.shape}")

In [ ]:
# 3. Build the RNN Model
model = Sequential([
    Input(shape=(x_train.shape[1], 1)),
    SimpleRNN(50, return_sequences=True), # return_sequences=True to pass to next RNN layer
    Dropout(0.2),
    
    SimpleRNN(50, return_sequences=False),
    Dropout(0.2),
    
    Dense(25, activation='relu'),
    Dense(1) # Final prediction
])

model.summary()

In [ ]:
# Compile the model
model.compile(optimizer='adam', loss='mean_squared_error')

In [ ]:
# 4. Train the Model
history = model.fit(
    x_train, y_train, 
    batch_size=32, 
    epochs=20
)

In [ ]:
# 5. Create the testing dataset
test_data = scaled_data[training_data_len - 60:, :]

# Create x_test and y_test
x_test = []
y_test = data[training_data_len:, :] # Actual unscaled values

for i in range(60, len(test_data)):
    x_test.append(test_data[i-60:i, 0])

# Convert to numpy array and reshape
x_test = np.array(x_test)
x_test = np.reshape(x_test, (x_test.shape[0], x_test.shape[1], 1))

In [ ]:
# 6. Evaluate and Predict
predictions = model.predict(x_test)

# Unscale the predictions so we can compare them to original prices
predictions = scaler.inverse_transform(predictions)

# Calculate Root Mean Squared Error (RMSE)
rmse = np.sqrt(np.mean(((predictions - y_test) ** 2)))
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")

In [ ]:
# 7. Plot the Predictions vs Actual Data
# Correctly extract the 'Close' column by using .xs for MultiIndex
close_data = df.xs('Close', level='Price', axis=1)
close_data.columns = ['Close'] # Rename the column from 'GOOG' to 'Close'

train = close_data[:training_data_len]
valid = close_data[training_data_len:].copy()
valid['Predictions'] = predictions

plt.figure(figsize=(16, 6))
plt.title('Google Stock Price Prediction - RNN')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Close Price USD ($)', fontsize=12)
plt.plot(train['Close'])
plt.plot(valid[['Close', 'Predictions']])
plt.legend(['Train', 'Validation (Actual)', 'Predictions'], loc='lower right')
plt.show()